# Lakebase & Apps — DAIS 2026 Runbook

Story: the ops manager doesn't just want to chat once. They want a live app, persistent history, and real-time data — all inside Databricks, no external services. And when they see a problem, they want to act on it immediately, right there.

**Message:** Lakebase is PostgreSQL, serverless, built into the platform. Databricks Apps are full-stack. Together they close the loop from data to production application — and from AI decision to operational action — deployed from a single `databricks.yml`.

> Docs: [Databricks Apps](https://docs.databricks.com/en/dev-tools/databricks-apps/index.html) · [Lakebase](https://docs.databricks.com/en/database/lakebase/index.html) · [Asset Bundles](https://docs.databricks.com/en/dev-tools/bundles/index.html) · [DevHub](https://developer.databricks.com)

## Pre-flight

- `all` target deployed and job completed
- Casper's Ops Dashboard app is in `RUNNING` state
- Refund Manager app is in `RUNNING` state
- Lakebase project `<catalog>-caspers` is up
- Complaint stream has run at least once (rows in `{catalog}.complaints.complaint_responses`)

Run the next cell for URLs.

In [ ]:
# Pre-flight: print fresh URLs for the demo.

try:
    CATALOG = dbutils.widgets.get("CATALOG") or "oleksandra"
except Exception:
    CATALOG = "oleksandra"

import json, re
from databricks.sdk import WorkspaceClient

w = WorkspaceClient()
host = w.config.host.rstrip("/")

print(f"Catalog: {CATALOG}")
print(f"Host:    {host}\n")

# Ops Dashboard app
print("Databricks Apps")
print("-" * 80)
try:
    df = spark.sql(f"SELECT resource_data FROM {CATALOG}._internal_state.resources WHERE resource_type = 'apps' ORDER BY created_at DESC")
    seen = set()
    for row in df.collect():
        info = json.loads(row.resource_data)
        name = info.get("name") or info.get("app_name", "")
        url  = info.get("url") or info.get("app_url", "")
        if name and name not in seen:
            seen.add(name)
            print(f"  {name}: {url}")
except Exception as e:
    print(f"  Could not read apps from uc_state: {e}")

# Refund Manager app
print()
# Name must match stages/apps.ipynb: APP_NAME = f"refundmanager-{CATALOG}" (no hyphen
# between "refund" and "manager") — Databricks Apps slug is capped at 30 chars and
# stages/apps.ipynb dropped the hyphen to leave room for longer catalog names.
refund_app_name = re.sub(r'-+', '-', re.sub(r'[^a-z0-9-]', '-', f'refundmanager-{CATALOG}'.lower())).strip('-')[:30]
try:
    ep = w.apps.get(refund_app_name)
    url = getattr(ep, 'url', None) or f"(deploy {refund_app_name} first)"
    print(f"Refund Manager app: {url}")
except Exception:
    print(f"Refund Manager app: not found — deploy apps/refund-manager with target=all")

# Lakebase project
print("\nLakebase")
print("-" * 80)
try:
    project_id = re.sub(r'[^a-z0-9-]', '-', f"{CATALOG}-caspers".lower())
    ep = w.postgres.get_endpoint(name=f"projects/{project_id}/branches/production/endpoints/primary")
    host_pg = ep.status.hosts.host if ep.status and ep.status.hosts else "?"
    state = str(ep.status.current_state) if ep.status else "?"
    print(f"  project:   {project_id}")
    print(f"  endpoint:  {host_pg}  [{state}]")
    print(f"  databases: caspers_ops  caspers_refund  caspers_complaint")
except Exception as e:
    print(f"  Could not read Lakebase project: {e}")

# Complaint decisions sanity check
print()
try:
    count = spark.sql(f"SELECT COUNT(*) AS n FROM {CATALOG}.complaints.complaint_responses").collect()[0]['n']
    print(f"complaint_responses rows: {count} {'✅' if count > 0 else '⚠️  run complaint stream first'}")
except Exception as e:
    print(f"complaint_responses: {e}")


## The demo

### 1 · Casper's Ops Dashboard

Open the app URL from the pre-flight cell.

It's a full-stack app: FastAPI backend, single-page frontend, deployed as a Databricks App — no external hosting, no separate infra. The app runs inside the workspace, inherits the caller's identity, and has direct access to the data.

Ask a question in the chat:
```
Which location needs the most attention right now?
```
The chat streams from the Supervisor Agent (same one from the Agent Bricks flow). Close the tab and reopen it — the conversation is still there. Chat history is persisted in **Lakebase**, not in browser storage.

Point at the sidebar: previous sessions are listed. Each session is a row in a Postgres table. Any team member with app access sees the same history.

```
What were the top complaints last week at that location?
```
```
Summarize what we know about the regulatory risk across all locations.
```

Flip to the **Revenue** and **Locations** tabs — live SQL warehouse queries, same UC-governed data the Genie spaces use, surfaced in a custom UI.

---

### 2 · Lakebase — PostgreSQL, serverless, built in

Open a SQL notebook. Query the session data directly:

```sql
SELECT session_id, created_at, title FROM caspers_ops.public.sessions ORDER BY created_at DESC LIMIT 10
```
```sql
SELECT role, LEFT(content, 120) AS preview FROM caspers_ops.public.messages WHERE session_id = '<id>' ORDER BY created_at
```

This is a real Postgres database — `psycopg`, `psql`, any Postgres client works. No connection string to manage; credentials are generated on demand via the SDK (`w.postgres.generate_database_credential`).

Message: one project, three logical databases (`caspers_ops`, `caspers_refund`, `caspers_complaint`), one endpoint, serverless. You don't provision it; you don't patch it; you don't think about it.

---

### 3 · How this was built — DABs + AppKit + DevHub + GitHub

The whole stack — app, job, warehouse, dashboards, Lakebase project — is declared in a single [`databricks.yml`](https://github.com/databricks-solutions/caspers-kitchens/blob/main/databricks.yml).

```bash
databricks bundle deploy -t all
databricks bundle run caspers
```

That's it. No ClickOps. No manual wiring. A new team member clones the repo and gets an identical environment.

**How the app was scaffolded:** [AppKit](https://developer.databricks.com) gives you a FastAPI + React starter wired to Databricks auth, UC, and the Apps runtime — the same scaffold Casper's Ops Dashboard started from. Start there, add your routes and components, drop it in `apps/` and reference it from `databricks.yml`.

**Where to start:** [DevHub](https://developer.databricks.com) — tutorials, templates, and reference apps for building on Databricks. The Casper's Kitchens source is on [GitHub](https://github.com/databricks-solutions/caspers-kitchens) — fork it, strip it down, or use it as a reference.

Key links:
- [Databricks Apps docs](https://docs.databricks.com/en/dev-tools/databricks-apps/index.html)
- [Asset Bundles docs](https://docs.databricks.com/en/dev-tools/bundles/index.html)
- [Lakebase docs](https://docs.databricks.com/en/database/lakebase/index.html)
- [DevHub](https://developer.databricks.com)
- [AppKit](https://developer.databricks.com/app-kit)
- [Casper's Kitchens on GitHub](https://github.com/databricks-solutions/caspers-kitchens)

## Complaints & Refunds — AI-driven ops loop

### 4 · Complaint Decisions

The **📬 Review Complaints** button is always visible in the action bar — no need to prompt for it.

Click it. The app fetches the latest decisions from `{catalog}.complaints.complaint_responses` — the output table of the complaint agent streaming worker — and renders them as a list:

- Each row shows: **category** · **rationale preview** · **decision badge** (🔴 Escalate / 🟢 Credit) · **credit amount**
- Click any row to expand it: full rationale, customer response message, order ID, and timestamp

This is AI-generated triage running continuously in the background — the ops manager sees the output without having to trigger anything.

---

### 5 · From Complaint to Refund

For rows where the agent recommended a credit, the expanded detail shows a gold **💳 Process Refund for $X.XX** button.

Click it. The Refund Manager modal opens with the **order ID pre-filled** from the complaint record.

Click **Evaluate** — the Refund Agent runs against the order, pulls live order data via UC functions, and returns a structured decision: refund class (none / partial / full), amount, and reasoning. The result appears inline as a card in the chat.

After the refund is evaluated, the complaint list **silently refreshes** — no page reload, no second click. The ops manager sees the updated triage inline.

**The story:** the complaint agent already did the triage. The ops manager reviews, confirms, and the refund agent closes the loop — all from one screen, no context switching.

---

### 6 · Refund Manager app

For high-volume refund processing, point to the standalone **Refund Manager** app (URL from pre-flight). It's purpose-built for working through a queue of orders, with the same refund agent under the hood.

## Between runs

- Apps stay deployed — no restart needed between slots.
- Lakebase endpoint stays up; databases and chat history persist across runs.
- Complaint stream runs on a cron — rows accumulate automatically. If the list is empty, trigger one manual run of the `complaint_agent_stream` job task.
- If the app is slow on first load it may be cold-starting — open it a minute before going on stage.
- Pre-flight shows no app: `Operational_App` task hasn't run. Re-run from the `caspers` job (`-t all`).

**Source of truth:** `stages/operational_app.ipynb`, `stages/lakebase_project.ipynb`, `stages/operational_lakebase.ipynb`, `stages/complaint_agent_stream.ipynb`.